In [1]:
import os
import pandas as pd
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing hemolytic peptide datasets (Almotairi et al.)

This notebook constructs a curated hemolytic peptide dataset from raw files associated with **Almotairi et al.**. It merges multiple CSV sources (e.g., `combined`, `hlppredfuse`, and `rnnamp`) into a standardized table, performs sequence normalization, separates potentially modified sequences, and runs duplicate/consistency checks before exporting final artifacts.

- **Toxic effect / endpoint:** hemolytic
- **Source:** Almotairi et al.
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset.

The pipeline performs the following steps:

- **Loads raw datasets** from:
  - `combined.csv`
  - `hlppredfuse.csv`
  - `rnnamp.csv`
- **Flags potentially modified sequences** using a simple rule-based detector:
  - presence of tokens like `AMD` or `ACT`, and/or
  - cases where `text` contains an `original_sequence` reference (suggesting modifications or annotation artifacts).
- **Creates two working datasets**:
  - a main dataset with normalized `sequence` and `label`,
  - a separate dataset for **modified** sequences.
- **Deduplicates by sequence** and resolves duplicates when labels are consistent:
  - **unique sequences** are kept as-is,
  - **duplicate sequences with the same label** are collapsed,
  - **duplicate sequences with conflicting labels** are exported as errors for manual review.
- **Builds metadata** from the project-wide Excel metadata sheet and adds QC statistics.
- **Exports results** (CSV + JSON) to the project `PATH_EXPORT` folder.

In [2]:
name_source = "Almotairi et al."
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
df_combined = pd.read_csv(f"{PATH_INPUT}/{name_source}/combined.csv")
df_hlppredfuse = pd.read_csv(f"{PATH_INPUT}/{name_source}/hlppredfuse.csv")
df_rnnamp = pd.read_csv(f"{PATH_INPUT}/{name_source}/rnnamp.csv")

- Concatenating dataset

In [4]:
df_almotairietal = pd.concat(
    [df_combined, df_hlppredfuse, df_rnnamp],
    ignore_index=True
)

In [5]:
pattern = r'^(A M D|A C T)\b|\b(A M D|A C T)$'

df_almotairietal["is_modified"] = (
    df_almotairietal["text"].str.contains(pattern, regex=True, na=False)
    |
    df_almotairietal.apply(
        lambda row: (
            pd.notna(row["original_sequence"]) and
            pd.notna(row["text"]) and
            row["original_sequence"] in row["text"]
        ),
        axis=1
    )
)

/tmp/ipykernel_25311/3677600650.py:4: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_almotairietal["text"].str.contains(pattern, regex=True, na=False)


In [6]:
modified_df = df_almotairietal[df_almotairietal["is_modified"]]
modified_df = (
    modified_df
    .rename(columns={"text": "sequence", "labels": "label"})
    .assign(
        sequence=lambda d: d["sequence"].astype(str).str.replace(" ", "", regex=False)
    )
    [["sequence", "label"]]
)

In [7]:
df_almotairietal.loc[~df_almotairietal["is_modified"], "original_sequence"] = df_almotairietal["text"]

In [8]:
df_almotairietal = (
    df_almotairietal
    .rename(columns={"original_sequence": "sequence", "labels": "label"})
    .assign(
        sequence=lambda d: d["sequence"].astype(str).str.replace(" ", "", regex=False)
    )
    [["sequence", "label"]]
)
df_almotairietal.shape

(13254, 2)

- Checking duplicates

In [9]:
df_remove_duplicated, df_errors, df_unique = processing_duplicated(df_almotairietal, group_seq="sequence", sort_key="label")

In [10]:
df_remove_duplicated_mod, df_errors_mod, df_unique_mod = processing_duplicated(modified_df, group_seq="sequence", sort_key="label")

In [11]:
df_full = pd.concat([df_unique, df_remove_duplicated], axis=0)
df_full.shape

(5400, 2)

In [12]:
df_full_mod = pd.concat([df_unique_mod, df_remove_duplicated_mod], axis=0)
df_full_mod.shape

(2322, 2)

- Working with metada

In [13]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

In [14]:
raw_total_sequences = (len(df_combined) + len(df_hlppredfuse) + len(df_rnnamp))

In [15]:
dict_metadata.update({
    "number_of_raw_sequences": int(raw_total_sequences),
    "number_of_sequences_retained": len(df_full),
    "number_of_positive_sequences": int((df_full["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full["label"] == 0).sum()),
    "number_of_erroneous_sequences": int(len(df_errors)),
    "number_of_modified_sequences": int(len(df_full_mod)),
    "number_of_erroneous_modified_sequences": int(len(df_errors_mod)),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'No information',
 'year of publication': 2024,
 'last update date': datetime.datetime(2024, 6, 28, 0, 0),
 'download date': Timestamp('2024-08-09 00:00:00'),
 'file format': 'csv',
 'peptide property': 'hemolytic, toxic',
 'dataset information': 'Positive, Negative',
 'unit of measurement': 'No information',
 'obtaining negative dataset': 'Previously published model dataset',
 'repository or server': 'https://github.com/mohamedelhakim/Transformer-CNN-Architecture/tree/main/data',
 'publication': 'https://www.nature.com/articles/s41598-024-63446-5',
 'number_of_raw_sequences': 13254,
 'number_of_sequences_retained': 5400,
 'number_of_positive_sequences': 1907,
 'number_of_negative_sequences': 3493,
 'number_of_erroneous_sequences': 121,
 'number_of_modified_sequences': 2322,
 'number_of_erroneous_modified_sequences': 0,
 'modified_sequences_included': False}

- Exporting data

In [16]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [17]:
df_full.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_hemolytic_dataset.csv", index=False)
modified_df.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/modified_hemolytic_dataset.csv", index=False)
df_errors.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/detected_error_sequences.csv", index=False)
df_errors_mod.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/detected_error_modified_sequences.csv", index=False)